# move_test

### 初始化

In [ ]:
# ros接口
import rospy
from sensor_msgs.msg import JointState

NODE_NAME = "airbot_test"
rospy.init_node(NODE_NAME)

In [ ]:
product_name = "airbot_play"
name_space = "/" + product_name
airbot_player_joint_names = ["joint{}".format(x) for x in range(1, 7)]
airbot_player_joint_nums = len(airbot_player_joint_names)
moveit_joint_state_pub = rospy.Publisher(
    name_space + "/joint_states", JointState, queue_size=2
)
moveit_joint_state_buf = JointState()
moveit_joint_state_buf.name = airbot_player_joint_names
moveit_joint_state_buf.velocity = [0 for _ in range(airbot_player_joint_nums)]
moveit_joint_state_buf.effort = [0 for _ in range(airbot_player_joint_nums)]
moveit_joint_state_buf.header.frame_id = product_name
moveit_joint_state_buf.position = [0, -1.5, 3, 1.2, 0, 0]
moveit_joint_state_buf.header.stamp = rospy.get_rostime()
moveit_joint_state_pub.publish(moveit_joint_state_buf)

In [ ]:
from airbot_play_control.control import RoboticArmAgent, ChooseGripper
import rospy
import math

# ROS节点初始化
NODE_NAME = "airbot_play_control_example"
rospy.init_node(NODE_NAME)
other_config = None
sim_player_nums = rospy.get_param("/airbot_play/sim_player_nums", default=0)
sim_type = ["real", "gazebo"][sim_player_nums]
print("Use", sim_type)
gripper = ["gripper", "inspire"][0]
if sim_type == "gazebo":
    other_config = ("", "airbot_play_arm")
gripper_controller = ChooseGripper(sim_type, gripper)()
arm = RoboticArmAgent(
    node_name=NODE_NAME, gripper=(4, gripper_controller), other_config=other_config
)

### xyz移动

In [ ]:
# x轴移动来回
arm.go_to_shift_single_axis_target(0, 0.1, vel_limit=1.0, acc_limit=1.0)
rospy.sleep(4)
arm.go_to_shift_single_axis_target(0, -0.1, vel_limit=1.0, acc_limit=1.0)

In [ ]:
# y轴移动来回
arm.go_to_shift_single_axis_target(1, 0.1, vel_limit=1.0, acc_limit=1.0)
rospy.sleep(4)
arm.go_to_shift_single_axis_target(1, -0.2, vel_limit=1.0, acc_limit=1.0)
rospy.sleep(2)
arm.go_to_shift_single_axis_target(1, 0.1, vel_limit=1.0, acc_limit=1.0)

In [ ]:
# z轴移动来回
arm.go_to_shift_single_axis_target(2, -0.05, vel_limit=1.0, acc_limit=1.0)
rospy.sleep(4)
arm.go_to_shift_single_axis_target(2, 0.1, vel_limit=1.0, acc_limit=1.0)
rospy.sleep(4)
arm.go_to_shift_single_axis_target(2, -0.05, vel_limit=1.0, acc_limit=1.0)

### rpy转动

In [ ]:
# r轴转动来回
arm.go_to_shift_single_axis_target(3, math.pi / 4, vel_limit=1.0, acc_limit=1.0)
rospy.sleep(4)
arm.go_to_shift_single_axis_target(3, -math.pi / 4, vel_limit=1.0, acc_limit=1.0)

In [ ]:
# p轴转动来回
arm.go_to_shift_single_axis_target(4, -math.pi / 4, vel_limit=1.0, acc_limit=1.0)
rospy.sleep(4)
arm.go_to_shift_single_axis_target(4, math.pi / 4, vel_limit=1.0, acc_limit=1.0)

In [ ]:
# y轴转动来回
arm.go_to_shift_single_axis_target(5, math.pi / 2, vel_limit=1.0, acc_limit=0.2)
rospy.sleep(4)
arm.go_to_shift_single_axis_target(5, -math.pi / 2, vel_limit=1.0, acc_limit=0.2)

### 退出

In [ ]:
# 关闭当前控制节点
arm.go_to_named_or_joint_target("Home", sleep_time=3)
rospy.signal_shutdown("end")